Importing Liberaries

In [27]:
import pandas as pd
import numpy as np
import random
from datetime import time
import uuid
import sqlalchemy as sa

Loading Data

In [28]:
df=pd.read_csv("swiggy_instamart_raw_data.csv")

In [29]:
df.head()

,Order_ID,Order_Date,Customer_ID,Customer_Name,City,Pincode,Product_ID,Product_Name,Category,Quantity,Unit_Price,Total_Amount,Discount,Final_Amount,Payment_Mode,Delivery_Time_Minutes,Delivery_Status,Rating,Review,Delivery_Partner
0,94317d2b-cba7-478a-b150-872fefaaedfb,2025-02-08,ba446cab-38d3-4724-ada4-7d0ebc03cd27,David Thompson,Delhi,NaN,8e8aa819-4739-4fcf-bb8a-ca84b3389b6b,Cola,Personal Care,-1.0,50.0,-50.0,10.0,-60.0,UPI,15.0,Cancelled,6.0,NaN,Kayla
1,b6763b8b-66e6-4a4b-ba0b-27c61c08c1e8,2025-02-28,1c77b5be-2336-4a6c-8ce3-f94f92464a0e,Daniel Mccarthy,Hyderabad,78652.0,fa8485aa-a169-4d37-b515-d72a93929d19,Biscuits,Vegetables,5.0,20.0,100.0,0.0,100.0,Cash,60.0,Cancelled,NaN,NaN,Sarah
2,57aa9bbb-789f-4896-ba26-df2a92d0acc6,2025-05-14,a74f0d22-4c5e-4cda-b54e-1871b188b18a,Christine Carter,NaN,95451.0,0df90c28-e3ac-48e3-ada7-151e586a18ce,cola,Dairy,1.0,100.0,100.0,-3.0,103.0,UPI,30.0,Delivered,2.0,To smile amount result.,John
3,10522989-dd4e-4239-9ea8-8c0770ccc15c,2025-02-26,2556e445-cae3-458c-9834-9d7a2821eaae,Daniel Garcia,Bangalore,65375.0,3c690a5b-7306-4f97-b5e3-31a8b23b9c0b,Shampoo,Fruits,NaN,20.0,NaN,-3.0,NaN,Cash,30.0,Delivered,5.0,NaN,Dawn
4,NaN,2024-12-27,1a042aa5-3e25-4a58-8943-15be0b557e12,Richard Sanchez,Hyderabad,82221.0,d6cacf86-e240-42d5-aa61-e3046090c2db,Soap,Beverages,1.0,100.0,100.0,NaN,NaN,Cash,NaN,Pending,5.0,NaN,Suzanne


Cleaning Text format

In [30]:
df['Product_Name'] = df['Product_Name'].str.strip().str.capitalize()
df['City'] = df['City'].str.strip().str.capitalize()
df['Delivery_Status'] = df['Delivery_Status'].str.strip().str.capitalize()
df['Payment_Mode'] = df['Payment_Mode'].str.strip().str.capitalize()
df['Order_Date'] = pd.to_datetime(df['Order_Date']).dt.date

Changing Negative Values to Positive

In [31]:
cols_to_fix = ['Quantity', 'Unit_Price', 'Discount', 'Total_Amount', 'Final_Amount','Delivery_Time_Minutes']
for i in cols_to_fix:
    df[i]=df[i].abs()

Assigning Right Category to Products

In [32]:
product_category_map = {
    'Milk': 'Dairy',
    'Biscuits': 'Snacks',
    'Cola': 'Beverages',
    'Shampoo': 'Personal Care',
    'Toothpaste': 'Personal Care',
    'Onion': 'Vegetables',
    'Apple': 'Fruits',
    'Bread': 'Grocery',
    'Soap': 'Personal Care'
}
df['Category']=df['Product_Name'].map(product_category_map)

Handling Nulls

In [33]:
df['Product_Name']=df['Product_Name'].fillna(random.choice(['Milk','Biscuits','Cola','Shampoo','Toothpaste','Onion','Apple','Bread','Soap']))
df['City']=df['City'].fillna(random.choice(['Delhi','Hyderabad','Mumbai','Chennai','Kolkata','Bangalore']))
df['Delivery_Status']=df['Delivery_Status'].fillna(random.choice(['Cancelled','Delivered','Pending']))
df['Payment_Mode']=df['Payment_Mode'].fillna(random.choice(['Upi','Card','Cash','Wallet']))
df.loc[df['Pincode'].isna(), 'Pincode'] = np.random.randint(1000, 99999, size=df['Pincode'].isna().sum())
df['Quantity'] = df['Quantity'].fillna(1)
df['Delivery_Time_Minutes'] = df['Delivery_Time_Minutes'].apply(lambda x: random.randint(15, 60) if pd.isna(x) else x)
df['Rating'] = df['Rating'].apply(lambda x: random.randint(1,6) if pd.isna(x) else x)
df['Order_ID'] = df['Order_ID'].apply(lambda x: str(uuid.uuid4()) if pd.isna(x) else x)
df['Customer_ID'] = df['Customer_ID'].apply(lambda x: str(uuid.uuid4()) if pd.isna(x) else x)
df['Product_ID'] = df['Product_ID'].apply(lambda x: str(uuid.uuid4()) if pd.isna(x) else x)

In [34]:
df['Quantity'] = df['Quantity'].astype(int)

Removing duplicates id's

In [35]:
df = df.drop_duplicates(subset='Order_ID')
df = df.drop_duplicates(subset='Product_ID')
df = df.drop_duplicates(subset='Pincode')

Finding Avg_unit_price of each product

In [36]:
products =['Milk','Biscuits','Cola','Shampoo','Toothpaste','Onion','Apple','Bread','Soap']
for product in products:
    avg_unit_price=df[df['Product_Name']==product]['Unit_Price'].mean()
    print(f"avg unit price of {product} is {avg_unit_price}" )

avg unit price of Milk is 37.7859778597786
avg unit price of Biscuits is 36.09987357774968
avg unit price of Cola is 37.25786924939467
avg unit price of Shampoo is 36.25802310654686
avg unit price of Toothpaste is 37.77845777233782
avg unit price of Onion is 37.03703703703704
avg unit price of Apple is 39.18457648546144
avg unit price of Bread is 36.09363008442057
avg unit price of Soap is 37.247652582159624


Filling Null values in Unit_price Column with Avg Unit Price of Product

In [37]:
avg_price_dict = {
    'Milk': 38,
    'Biscuits': 37,
    'Cola': 37,
    'Shampoo': 37,
    'Toothpaste': 38,
    'Onion': 38,
    'Apple': 40,
    'Bread': 35,
    'Soap': 38
}
df['Unit_Price'] = df.apply(
    lambda row: avg_price_dict[row['Product_Name']] if pd.isnull(row['Unit_Price']) and row['Product_Name'] in avg_price_dict else row['Unit_Price'],
    axis=1
)
df['Unit_Price'] = df['Unit_Price'].astype(int)

Filling Null values in Category acc to the Product

In [38]:
product_category_map = {
    'Milk': 'Dairy',
    'Biscuits': 'Snacks',
    'Cola': 'Beverages',
    'Shampoo': 'Personal Care',
    'Toothpaste': 'Personal Care',
    'Onion': 'Vegetables',
    'Apple': 'Fruits',
    'Bread': 'Grocery',
    'Soap': 'Personal Care'
}
df['Category'] = df['Category'].fillna(df['Product_Name'].map(product_category_map))

Creating a new column of Cost Price and Total Purchased Amount

In [39]:
df['Cost_Price'] = df['Unit_Price'].apply(
    lambda x: round(x - random.uniform(1, 2)) if pd.notna(x) and x < 9 else
              round(x - random.uniform(1, 5)) if x <= 49 else
              round(x - random.uniform(5, 15)) if pd.notna(x)
              else None
)
df['Total_Purchased_Amount']=df['Cost_Price'] * df['Quantity']
df['Total_Purchased_Amount'] = df['Total_Purchased_Amount'].astype(int)

Filling null values in Total amount by calculating

In [40]:
df=df.drop('Total_Amount',axis=1)
df['Total_sold_amount'] = df['Unit_Price'] * df['Quantity']
df['Total_sold_amount'] = df['Total_sold_amount'].astype(int)

Handling The null values in discount

In [41]:
df['Discount'] = df['Discount'].fillna(0)
df['Discount'] = df.apply(lambda row: round(random.uniform(0.00, 0.20) * (row['Total_sold_amount'] - row['Total_Purchased_Amount']))
    if pd.notna(row['Total_sold_amount']) and pd.notna(row['Total_Purchased_Amount']) else None,
    axis=1
)

Calculating Correct Final Amount

In [42]:
df=df.drop('Final_Amount',axis=1)
df['Final_sold_amount'] = df['Total_sold_amount'] - df['Discount']
df['Final_sold_amount'] = df['Final_sold_amount'].astype(int)
df['Rating'] = df['Rating'].astype(int)

Creating a new column of Profit

In [43]:
df['Profit']=df['Final_sold_amount']-df['Total_Purchased_Amount']
df['Profit'] = df['Profit'].astype(int)

Creating a new column for order time with random values

In [44]:
hours = np.random.randint(0, 24, size=len(df))
minutes = np.random.randint(0, 60, size=len(df))
seconds = np.random.randint(0, 60, size=len(df))
df['Order_Time'] = [time(h, m, s) for h, m, s in zip(hours, minutes, seconds)]
df['Delivery_Time_Minutes'] = df['Delivery_Time_Minutes'].astype(int)

Creating a new column of region acc to cities

In [45]:
city_region_dict = {
    'Delhi': 'North',
    'Mumbai': 'West',
    'Kolkata': 'East',
    'Chennai': 'South',
    'Bangalore': 'South',
    'Hyderabad': 'South',
}
df['Region']=df['City'].map(city_region_dict)
city_state_dict = {
    'Delhi': 'Delhi',
    'Mumbai': 'Maharastra',
    'Kolkata': 'West Bengal',
    'Chennai': 'Tamil Naidu',
    'Bangalore': 'Karnataka',
    'Hyderabad': 'Telangana',
}
df['State']=df['City'].map(city_state_dict)


Deleting the Review column as it is not necessary

In [46]:
df=df.drop('Review',axis=1)

Now deleting rows with remaning null values

In [47]:
df=df.dropna()

Connecting This datafrome to the SQL server

In [48]:
server = 'localhost\\SQLEXPRESS'
database = 'SWIGGY'  
driver = 'ODBC Driver 17 for SQL Server'

connection_string = f'mssql+pyodbc://@{server}/{database}?trusted_connection=yes&driver={driver}'
engine = sa.create_engine(connection_string)


In [49]:
df.to_sql('instamart_orders', con=engine, if_exists='replace', index=False)


82

In [50]:
df.describe()


,Pincode,Quantity,Unit_Price,Discount,Delivery_Time_Minutes,Rating,Cost_Price,Total_Purchased_Amount,Total_sold_amount,Final_sold_amount,Profit
count,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000,8216.000000
mean,49877.359786,3.316091,37.118914,1.625365,45.283106,3.495131,32.048685,106.279576,123.139727,121.514362,15.234786
std,28855.844285,3.098232,31.773546,2.933818,34.439958,1.698942,28.929651,165.071618,185.367783,183.316769,21.852318
min,519.000000,1.000000,5.000000,0.000000,10.000000,1.000000,3.000000,3.000000,5.000000,5.000000,1.000000
25%,24900.000000,1.000000,10.000000,0.000000,15.000000,2.000000,7.000000,16.000000,20.000000,20.000000,3.000000
50%,49853.500000,2.000000,35.000000,1.000000,38.000000,4.000000,31.000000,40.000000,50.000000,49.000000,8.000000
75%,74396.500000,5.000000,50.000000,2.000000,60.000000,5.000000,40.000000,105.000000,114.000000,113.000000,17.000000
max,99940.000000,10.000000,100.000000,28.000000,120.000000,6.000000,95.000000,950.000000,1000.000000,1000.000000,145.000000


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8216 entries, 0 to 9999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                8216 non-null   object 
 1   Order_Date              8216 non-null   object 
 2   Customer_ID             8216 non-null   object 
 3   Customer_Name           8216 non-null   object 
 4   City                    8216 non-null   object 
 5   Pincode                 8216 non-null   float64
 6   Product_ID              8216 non-null   object 
 7   Product_Name            8216 non-null   object 
 8   Category                8216 non-null   object 
 9   Quantity                8216 non-null   int64  
 10  Unit_Price              8216 non-null   int64  
 11  Discount                8216 non-null   int64  
 12  Payment_Mode            8216 non-null   object 
 13  Delivery_Time_Minutes   8216 non-null   int64  
 14  Delivery_Status         8216 non-null   objec